# Baselines: HistGradientBoosting, LightGBM and time weighting

Simple models on the **165 supplied features** only (no graph features), using the same chronological checks as the final model:

1. **HistGradientBoosting** with fixed settings;
2. **LightGBM** with round counts chosen by early stopping inside each fitting window;
3. **Time-weighted HistGradientBoosting**: recent rows get more weight (half-life 15 steps), the weighting used by the final model.

In [ ]:
from pathlib import Path
import json
import platform
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

# hide the LightGBM deprecation warning about eval_set
warnings.filterwarnings("ignore", message="The argument 'eval_set' is deprecated")

# add the data folder to this path.
DATA_FOLDER = "bt-4012-competition-2026"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / DATA_FOLDER).is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"Open this notebook inside the project containing {DATA_FOLDER}.")
DATA = ROOT / DATA_FOLDER
FEATURES = [f"feat_{i}" for i in range(1, 166)]
print("Python:", platform.python_version(), "scikit-learn:", sklearn.__version__, "LightGBM:", lgb.__version__)

Python: 3.13.1 scikit-learn: 1.7.1 LightGBM: 4.7.0


## Validation checks
All validation uses three chronological checks: (this is the same as the final submission's notebook)

| Check | Fit on known labels | Score on known labels | Horizon | Role |
|---|---|---|---|---|
| `steps_22_28` | steps 1–21 | steps 22–28 | 7 steps | Main check for **comparing** models; it has the most room between them. |
| `steps_29_35` | steps 1–28 | steps 29–35 | 7 steps | Closest to the test period; scores are usually very high, so it separates models less. |
| `14_step_check` | steps 1–21 (the **same** fitted model as `steps_22_28`) | steps 22–35 | 14 steps | Same horizon as the real task: fit ≤ 35, predict 36–49. |

Why the 14-step check usually scores lower than `steps_22_28`:
- **Longer horizon.** The model must rank transactions up to 14 steps after its last training step, and the far half (29–35) has drifted further from steps 1–21.
- **No refit at step 28.** The `steps_29_35` check refits on 1–28 before scoring 29–35; the 14-step check does not.
- **Pooled AUC across steps.** Scores from 14 different steps are ranked together, so any step-to-step shift in score level lowers AUC even when the ranking within each step is good.

Rules:
- Fit and score on **known labels only**; unknown rows never count as licit.
- Choose settings using the mean ROC-AUC of the two 7-step checks (`steps_22_28` and `steps_29_35`). Report all three checks; never use the 14-step check for selection.

In [2]:
CHECKS = [
    {"name": "steps_22_28", "train_start": 1, "train_end": 21, "validation_start": 22, "validation_end": 28},
    {"name": "steps_29_35", "train_start": 1, "train_end": 28, "validation_start": 29, "validation_end": 35},
    {"name": "14_step_check", "train_start": 1, "train_end": 21, "validation_start": 22, "validation_end": 35},
]
SELECTION_CHECKS = ["steps_22_28", "steps_29_35"]  # the 14-step check is reported, never used for selection
display(pd.DataFrame(CHECKS))

,name,train_start,train_end,validation_start,validation_end
0,steps_22_28,1,21,22,28
1,steps_29_35,1,28,29,35
2,14_step_check,1,21,22,35


## Read the training transactions
Blank labels are unknown and cannot enter fitting or scoring. `txId` and `time_step` identify rows and periods but are not features.

In [3]:
# load only the columns needed: IDs, time step, features, label
train = pd.read_csv(DATA / "train.csv", usecols=["txId", "time_step", *FEATURES, "label"])
# stop if any known label is not 0 or 1
unexpected = train.loc[train.label.notna() & ~train.label.isin([0, 1]), "label"].unique()
if len(unexpected):
    raise ValueError(f"Unexpected known labels: {unexpected}")
assert train.txId.is_unique, "Duplicate training transaction IDs need investigation."
assert np.isfinite(train[FEATURES].to_numpy()).all(), "Missing/infinite training features."
# labelled rows only: the only rows models fit on or score
known = train.loc[train.label.isin([0, 1])].copy()
known["label"] = known["label"].astype(int)
print(f"Training transactions: {len(train):,}; known labels: {len(known):,}; "
      f"unknown labels excluded: {train.label.isna().sum():,}")
display(known.label.value_counts().rename_axis("label").to_frame("transactions"))

def check_rows(check):
    """Known-label fit and validation rows for one chronological check."""
    fit = known.loc[known.time_step.between(check["train_start"], check["train_end"])]
    valid = known.loc[known.time_step.between(check["validation_start"], check["validation_end"])]
    assert check["train_end"] < check["validation_start"], "Time periods overlap."
    assert set(fit.txId).isdisjoint(valid.txId), "Transaction IDs overlap."
    assert fit.label.nunique() == valid.label.nunique() == 2, "Both classes required."
    return fit, valid


def run_checks(fit_window, features=FEATURES):
    """Fit one model per fitting window and return class-1 probabilities for every check.

    steps_22_28 and 14_step_check share fitting steps 1–21, so they reuse one model."""
    # steps_22_28 and 14_step_check share the window 1–21 model, so it is fitted once
    fitted, frames = {}, []
    for check in CHECKS:
        fit, valid = check_rows(check)
        window = (check["train_start"], check["train_end"])
        if window not in fitted:
            fitted[window] = fit_window(fit, check)
        model = fitted[window]
        # probability of class 1 (illicit)
        p = model.predict_proba(valid[features])[:, list(model.classes_).index(1)]
        frames.append(pd.DataFrame({"check": check["name"], "txId": valid.txId.to_numpy(),
                                    "time_step": valid.time_step.to_numpy(),
                                    "label": valid.label.to_numpy(), "illicit_probability": p}))
    return pd.concat(frames, ignore_index=True), fitted


def score_checks(predictions, column="illicit_probability"):
    """ROC-AUC and average precision for each check."""
    rows = []
    for check in CHECKS:
        group = predictions.loc[predictions.check.eq(check["name"])]
        rows.append({"check": check["name"],
                     "fit_steps": f'{check["train_start"]}–{check["train_end"]}',
                     "validation_steps": f'{check["validation_start"]}–{check["validation_end"]}',
                     "validation_rows": len(group), "validation_illicit": int(group.label.sum()),
                     "validation_roc_auc": roc_auc_score(group.label, group[column]),
                     "average_precision": average_precision_score(group.label, group[column])})
    return pd.DataFrame(rows)


def per_step_auc(predictions, columns):
    """AUC for each time step of the two 7-step checks (steps with both classes only)."""
    rows = []
    sevens = predictions.loc[predictions.check.isin(SELECTION_CHECKS)]
    for (check, step), group in sevens.groupby(["check", "time_step"], sort=True):
        row = {"check": check, "time_step": int(step), "labelled_rows": len(group),
               "illicit_rows": int(group.label.sum())}
        for name, column in columns.items():
            row[name] = roc_auc_score(group.label, group[column]) if group.label.nunique() == 2 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

Training transactions: 141,772; known labels: 31,235; unknown labels excluded: 110,537


,transactions
label,
0,27591
1,3644


## 1. HistGradientBoosting
scikit-learn's `HistGradientBoostingClassifier` with fixed settings. Each fitting window gets a **new** model, scored on later labelled transactions. ROC-AUC and average precision use class-1 probabilities. Precision, recall, F1 and the confusion matrices use an untuned 0.5 threshold for interpretation; changing it cannot improve ROC-AUC.

In [4]:
THRESHOLD = 0.5  # Diagnostic classification metrics only.
HGB_PARAMS = dict(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.0,
    class_weight=None,
    early_stopping=False,
    random_state=42,
)


# same settings for every window; a fresh model is built each time
def fit_hgb(fit, check):
    return HistGradientBoostingClassifier(**HGB_PARAMS).fit(fit[FEATURES], fit.label)

# class-1 probabilities for every check
hgb_predictions, _ = run_checks(fit_hgb)
hgb_metrics = score_checks(hgb_predictions)
threshold_rows, confusions = [], {}
for check in CHECKS:
    group = hgb_predictions.loc[hgb_predictions.check.eq(check["name"])]
    # 0.5 threshold, only for precision, recall, F1 and the confusion matrix
    predicted = (group.illicit_probability >= THRESHOLD).astype(int)
    threshold_rows.append({
        "illicit_precision": precision_score(group.label, predicted, zero_division=0),
        "illicit_recall": recall_score(group.label, predicted, zero_division=0),
        "illicit_f1": f1_score(group.label, predicted, zero_division=0)})
    confusions[check["name"]] = pd.DataFrame(
        confusion_matrix(group.label, predicted, labels=[0, 1]),
        index=["Actual licit", "Actual illicit"], columns=["Predicted licit", "Predicted illicit"])
hgb_metrics = pd.concat([hgb_metrics, pd.DataFrame(threshold_rows)], axis=1)
with pd.option_context("display.max_columns", None):
    display(hgb_metrics.round(5))
for name, matrix in confusions.items():
    print(name)
    display(matrix)

,check,fit_steps,validation_steps,validation_rows,validation_illicit,validation_roc_auc,average_precision,illicit_precision,illicit_recall,illicit_f1
0,steps_22_28,1–21,22–28,5677,671,0.94792,0.88050,0.96007,0.78838,0.86579
1,steps_29_35,1–28,29–35,6028,1102,0.99694,0.98695,0.98643,0.92377,0.95408
2,14_step_check,1–21,22–35,11705,1773,0.89877,0.80969,0.96613,0.67569,0.79522


steps_22_28


,Predicted licit,Predicted illicit
Actual licit,4984,22
Actual illicit,142,529


steps_29_35


,Predicted licit,Predicted illicit
Actual licit,4912,14
Actual illicit,84,1018


14_step_check


,Predicted licit,Predicted illicit
Actual licit,9890,42
Actual illicit,575,1198


## 2. LightGBM
For each fitting window, the **last four steps inside that window** choose the number of boosting rounds by early stopping. A fresh model is then fitted on the whole window with that round count and scored on the later validation steps, so validation labels never enter fitting or early stopping.

In [5]:
BASE_PARAMS = dict(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=1.0,
    objective="binary",
    metric="auc",
    random_state=42,
    n_jobs=4,
    verbosity=-1,
)
EARLY_STOPPING_ROUNDS = 50
# inner-validation details for each window
selection_rows = []

def fit_lightgbm(fit, check):
    inner_valid_start = check["train_end"] - 3  # last four steps of the fitting window
    inner_train = fit.loc[fit.time_step < inner_valid_start]
    inner_valid = fit.loc[fit.time_step >= inner_valid_start]
    assert inner_train.time_step.max() < inner_valid.time_step.min()
    assert inner_train.label.nunique() == inner_valid.label.nunique() == 2
    # early stopping watches the inner validation steps only
    selector = lgb.LGBMClassifier(**BASE_PARAMS)
    selector.fit(inner_train[FEATURES], inner_train.label,
                 eval_set=[(inner_valid[FEATURES], inner_valid.label)], eval_metric="auc",
                 callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False)])
    # number of rounds chosen by early stopping
    best_rounds = int(selector.best_iteration_)
    assert 1 <= best_rounds <= BASE_PARAMS["n_estimators"]
    selection_rows.append({
        "train_end": check["train_end"],
        "inner_train_steps": f'{check["train_start"]}–{inner_valid_start - 1}',
        "inner_validation_steps": f'{inner_valid_start}–{check["train_end"]}',
        "inner_train_rows": len(inner_train), "inner_validation_rows": len(inner_valid),
        "selected_rounds": best_rounds,
        "inner_validation_auc": roc_auc_score(inner_valid.label,
                                              selector.predict_proba(inner_valid[FEATURES])[:, 1])})
    # refit on the whole fitting window with the chosen round count
    return lgb.LGBMClassifier(**{**BASE_PARAMS, "n_estimators": best_rounds}).fit(fit[FEATURES], fit.label)

lgbm_predictions, _ = run_checks(fit_lightgbm)
display(pd.DataFrame(selection_rows).round(5))
lgbm_metrics = score_checks(lgbm_predictions)
display(lgbm_metrics.round(5))

,train_end,inner_train_steps,inner_validation_steps,inner_train_rows,inner_validation_rows,selected_rounds,inner_validation_auc
0,21,1–17,18–21,16855,2675,315,0.99144
1,28,1–24,25–28,23606,1601,603,0.99835


,check,fit_steps,validation_steps,validation_rows,validation_illicit,validation_roc_auc,average_precision
0,steps_22_28,1–21,22–28,5677,671,0.95368,0.87912
1,steps_29_35,1–28,29–35,6028,1102,0.99664,0.98493
2,14_step_check,1–21,22–35,11705,1773,0.90104,0.80883


### Check performance by time step
The pooled ROC-AUC can hide a difficult period. Each row scores one time step with both classes present, for LightGBM and HistGradientBoosting on the **same validation transactions**.

In [6]:
# identifies the same validation rows in both models
KEYS = ["check", "txId", "time_step", "label"]
joined = lgbm_predictions.merge(hgb_predictions, on=KEYS, validate="one_to_one",
                                suffixes=("_lightgbm", "_hist_gradient_boosting"))
assert len(joined) == len(lgbm_predictions) == len(hgb_predictions)
with pd.option_context("display.max_rows", 30):
    display(per_step_auc(joined, {"auc_lightgbm": "illicit_probability_lightgbm",
                                  "auc_hist_gradient_boosting": "illicit_probability_hist_gradient_boosting"}).round(5))
comparison = lgbm_metrics[["check", "validation_roc_auc", "average_precision"]].merge(
    hgb_metrics[["check", "validation_roc_auc", "average_precision"]],
    on="check", validate="one_to_one", suffixes=("_lightgbm", "_hist_gradient_boosting"))
comparison["auc_change_vs_hist_gradient_boosting"] = (
    comparison.validation_roc_auc_lightgbm - comparison.validation_roc_auc_hist_gradient_boosting)
display(comparison.round(5))

,check,time_step,labelled_rows,illicit_rows,auc_lightgbm,auc_hist_gradient_boosting
0,steps_22_28,22,1763,158,0.99853,0.99940
1,steps_22_28,23,1187,53,0.95659,0.95538
2,steps_22_28,24,1126,137,0.77343,0.73561
3,steps_22_28,25,594,118,0.99954,0.99973
4,steps_22_28,26,517,96,0.93329,0.92609
5,steps_22_28,27,206,24,0.99931,0.99977
6,steps_22_28,28,284,85,1.00000,1.00000
7,steps_29_35,29,1174,329,0.99644,0.99721
8,steps_29_35,30,524,83,0.99404,0.99434
9,steps_29_35,31,710,106,0.98825,0.98943


,check,validation_roc_auc_lightgbm,average_precision_lightgbm,validation_roc_auc_hist_gradient_boosting,average_precision_hist_gradient_boosting,auc_change_vs_hist_gradient_boosting
0,steps_22_28,0.95368,0.87912,0.94792,0.88050,0.00576
1,steps_29_35,0.99664,0.98493,0.99694,0.98695,-0.00030
2,14_step_check,0.90104,0.80883,0.89877,0.80969,0.00227


## 3. Time-weighted HistGradientBoosting
The test set comes after every training step, so the most recent training patterns may matter most. This keeps every labelled row but gives **recent rows more weight**: a row 15 steps older than the newest fitting step counts half as much,

`weight = 0.5 ** ((latest_fit_step - row_step) / 15)`  (half-life of 15 steps).

Both models use the HistGradientBoosting settings above; only the sample weights differ. The blend share `w` of the time-weighted model is swept from 0% to 100% in 10% steps (`1 − w` is ordinary HGB) and chosen by the highest mean AUC over the two 7-step checks (ties go to the smaller share). The weights are computed inside each fitting window from that window's own latest step.

In [7]:
# a row's weight halves every 15 steps
HALF_LIFE = 15
SHARES = np.round(np.arange(0, 1.01, 0.1), 1)


# weight = 0.5 ** ((latest step - row step) / half-life)
def time_weights(steps, latest_step):
    return 0.5 ** ((latest_step - np.asarray(steps)) / HALF_LIFE)

print("Example weights when the latest fitting step is 35:")
display(pd.DataFrame({"time_step": [35, 30, 20, 5, 1],
                      "weight": time_weights([35, 30, 20, 5, 1], 35).round(3)}))


def fit_weighted(fit, check):
    return HistGradientBoostingClassifier(**HGB_PARAMS).fit(
        fit[FEATURES], fit.label, sample_weight=time_weights(fit.time_step, fit.time_step.max()))

# time-weighted model for every check
weighted, _ = run_checks(fit_weighted)
# pair ordinary and weighted predictions on the same rows
pair = hgb_predictions.merge(weighted, on=KEYS, validate="one_to_one", suffixes=("_ordinary", "_weighted"))
assert len(pair) == len(hgb_predictions) == len(weighted)
sweep_rows = []
# blend share of the time-weighted model
for w in SHARES:
    pair["blend"] = (1 - w) * pair.illicit_probability_ordinary + w * pair.illicit_probability_weighted
    row = {"time_weighted_share": w}
    for _, s in score_checks(pair, "blend").iterrows():
        row[s.check] = s.validation_roc_auc
    sweep_rows.append(row)
sweep = pd.DataFrame(sweep_rows)
sweep["mean_7_step"] = sweep[SELECTION_CHECKS].mean(axis=1)
# highest mean 7-step AUC; ties go to the smaller share
selected_share = float(sweep.sort_values(["mean_7_step", "time_weighted_share"],
                                          ascending=[False, True]).time_weighted_share.iloc[0])
display(sweep[["time_weighted_share", *[c["name"] for c in CHECKS], "mean_7_step"]].round(6))
print(f"Selected time-weighted share: {selected_share:.0%}")

Example weights when the latest fitting step is 35:


,time_step,weight
0,35,1.000
1,30,0.794
2,20,0.500
3,5,0.250
4,1,0.208


,time_weighted_share,steps_22_28,steps_29_35,14_step_check,mean_7_step
0,0.0,0.947922,0.996940,0.898772,0.972431
1,0.1,0.949332,0.996924,0.901041,0.973128
2,0.2,0.950404,0.996913,0.902951,0.973659
3,0.3,0.951305,0.996895,0.904665,0.974100
4,0.4,0.952097,0.996874,0.906198,0.974485
5,0.5,0.952768,0.996860,0.907614,0.974814
6,0.6,0.953383,0.996843,0.908950,0.975113
7,0.7,0.953941,0.996809,0.910247,0.975375
8,0.8,0.954464,0.996770,0.911524,0.975617
9,0.9,0.954966,0.996736,0.912835,0.975851


Selected time-weighted share: 100%


## Findings

**HistGradientBoosting**
1. ROC-AUC is 0.9479 on steps 22–28, 0.9969 on steps 29–35 and 0.8988 on the 14-step check.
2. Average precision is 0.8805 on steps 22–28 and 0.9870 on steps 29–35.
3. At the 0.5 threshold, illicit precision stays high (0.96–0.99) but recall falls from 0.92 (29–35) to 0.68 on the 14-step check: more illicit transactions are missed as the horizon grows.

**LightGBM**
1. Early stopping on the last four fitting steps chose 315 rounds (window 1–21) and 603 rounds (window 1–28); the final model uses their median, 459.
2. ROC-AUC is 0.9537 on steps 22–28, 0.9966 on steps 29–35 and 0.9010 on the 14-step check: +0.0058 and +0.0023 over HistGradientBoosting on 22–28 and the 14-step check, −0.0003 on 29–35.
3. Per step, step 24 is by far the hardest (AUC 0.7734 for LightGBM, 0.7356 for HistGradientBoosting); step 26 is the next weakest (about 0.93). Every step of 29–35 is above 0.98.

**Time weighting**
1. Time weighting helps the 22–28 and 14-step checks steadily as its share grows: the time-weighted model alone scores 0.95546 on steps 22–28 and 0.91421 on the 14-step check, against 0.94792 and 0.89877 for ordinary HGB.
2. On steps 29–35 every weighted share is slightly below ordinary HGB (0.99694 → 0.99668 at 100%); the differences are in the fourth decimal place.
3. The highest mean 7-step AUC is the **100% time-weighted** model (0.97607, against 0.97243 for ordinary HGB), so the rule selects it. The final model uses the same half-life-15 weights.
4. The equal 50/50 blend scores 0.95277 / 0.99686 / 0.90761.